# Fine-tune Qwen3 for the Minecraft agent (Unsloth QLoRA)

Trains on the samples written by `export-sft.js` and produces a LoRA adapter you download and turn into an Ollama model.

**Before running, in the right-hand settings panel:**
1. Accelerator: **GPU T4 x2** (not P100; Unsloth needs compute capability 7.0+). Only one T4 is used.
2. Internet: **On** (model and package downloads). Needs a phone-verified Kaggle account.
3. Add Data: attach your **private** dataset containing `train.jsonl`, `val.jsonl` (optional) and `train_sft.py`.

Then **Run All**. For a long run use *Save Version -> Save & Run All* so it keeps going if you close the tab.

In [ ]:
import glob, os, subprocess

# ---- settings you may want to change -----------------------------------------------------------------------------
MODEL = "unsloth/Qwen3-4B"      # unsloth/Qwen3-1.7B if you run out of GPU memory
MAX_SEQ_LEN = 12288             # T4 (16 GB): 12288 usually fits for 4B; lower it if you hit CUDA out-of-memory
EPOCHS = 2
OUT = "/kaggle/working/qwen3-4b-minecraft"
RUN_GGUF = False                # True: also build a q4_k_m GGUF at the end (slow here: it compiles llama.cpp)
# --------------------------------------------------------------------------------------------------------------------

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4; Unsloth does not support HF's two-GPU DataParallel mode
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise SystemExit(f"{pattern} not found under /kaggle/input. Attach your dataset (Add Data) and re-run.")
    return hits[0]

TRAIN_PY = find("train_sft.py")
DATA_DIR = os.path.dirname(find("train.jsonl"))
print("script:", TRAIN_PY)
print("data:  ", DATA_DIR, "->", os.listdir(DATA_DIR))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q unsloth

## Check the data first
Look at the `kept` / `dropped` numbers. If a lot is `too-long`, raise `MAX_SEQ_LEN` (if memory allows) or re-export with a smaller trim window / `--max-tokens`.

In [ ]:
!python "{TRAIN_PY}" --data "{DATA_DIR}" --model "{MODEL}" --max-seq-len {MAX_SEQ_LEN} --prepare-only

## Train

In [ ]:
!python "{TRAIN_PY}" --data "{DATA_DIR}" --out "{OUT}" --model "{MODEL}" --max-seq-len {MAX_SEQ_LEN} --epochs {EPOCHS}

## Package the adapter for download
After this cell finishes, open the notebook's **Output** tab (or `/kaggle/working`) and download `lora.zip`.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/lora", "zip", os.path.join(OUT, "lora"))
size = os.path.getsize("/kaggle/working/lora.zip") / 1e6
print(f"/kaggle/working/lora.zip  ({size:.0f} MB)")
shutil.rmtree(os.path.join(OUT, "checkpoints"), ignore_errors=True)   # keep the output small

## Optional: GGUF for Ollama
Set `RUN_GGUF = True` in the settings cell and re-run this cell. It reloads the saved adapter, merges it and writes a quantized GGUF
(builds llama.cpp, so it takes a while). You can also skip it and convert later on any machine.

In [ ]:
if RUN_GGUF:
    script = f'''
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(model_name="{OUT}/lora", max_seq_length={MAX_SEQ_LEN}, load_in_4bit=True)
model.save_pretrained_gguf("{OUT}/gguf", tok, quantization_method="q4_k_m")
'''
    open("/kaggle/working/make_gguf.py", "w").write(script)
    subprocess.run(["python", "/kaggle/working/make_gguf.py"], check=True)
    print(os.listdir(f"{OUT}/gguf"))
else:
    print("RUN_GGUF is False: skipped.")